# Step 13 · Retrieval-augmented generation (RAG)

Chunk documents, retrieve with BM25 / embeddings, answer with a LangChain chain, measure retrieval.

*Same as the dashboard page, done in code.*

In [ ]:
# ── Setup (run first) ────────────────────────────────────────────────────────────
# Uses the same lab registry as the dashboard. To experiment in a sandbox instead:
#   os.environ["LAB_HOME"] = "/tmp/my_lab"   (before importing lab)
import os, sys
ROOT = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
sys.path.insert(0, ROOT); os.chdir(ROOT)
os.environ.setdefault("HF_HUB_OFFLINE", "1")          # models are already cached locally
import pandas as pd
from lab import data, registry, paths, rewards
data.bootstrap_samples()
BASE = registry.ensure_base(paths.DEFAULT_BASE_MODEL)
if not registry.rewards():
    registry.add_reward(rewards.DEFAULT_RULES, note="Default draft rules")

def latest(kind):
    """Newest model id of a kind: base | sft | rl | rlhf | rm (or None)."""
    ms = [m for m in registry.models() if m["kind"] == kind]
    return ms[-1]["id"] if ms else None

def latest_ds(role):
    d = registry.latest_dataset(role)
    return d["id"] if d else None

print("Lab home:", paths.home(), "| base model:", BASE)

In [ ]:
from lab import rag
if "tool_store" not in rag.list_kbs():
    rag.build_sample(embed=True)
pd.DataFrame(rag.retrieve("tool_store", "Who approves a discount for an upset shopper?", k=3, method="embedding"))[["doc", "score", "text"]]

In [ ]:
rows = []
for m in ("bm25", "embedding", "hybrid"):
    _, mean = rag.eval_retrieval("tool_store", rag.sample_eval(), k=3, method=m)
    rows.append({"method": m, **{k: round(v, 3) for k, v in mean.items()}})
pd.DataFrame(rows)

In [ ]:
chain = rag.build_chain(latest("sft") or BASE)      # prompt | LabChatModel | StrOutputParser
q = "How long is the warranty on a drill battery?"
hits = rag.retrieve("tool_store", q, k=3, method="hybrid")
chain.invoke({"context": rag.format_context(hits), "question": q})